# Training Notebook: Classical ML (TF-IDF+LR)

This notebook trains the requirement-level binary classifier. Each training row contains a resume, a list of minimum requirements, and one binary label per requirement.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

PROJECT_ROOT

PosixPath('/Users/punpunpmbp/Documents/Documents/THU_Project/DL/semantic-resume_job-matching-system')

## Inspect Sample Data

In [2]:
import json
from pprint import pprint

sample_path = PROJECT_ROOT / "data" / "processed" / "resume_score_details.jsonl"
if not sample_path.exists():
    sample_path = PROJECT_ROOT / "data" / "raw" / "sample.jsonl"

with sample_path.open("r", encoding="utf-8") as f:
    first_record = json.loads(next(f))

pprint(first_record)

{'labels': [1, 1, 1],
 'minimum_requirements': ['Proficiency in conducting and managing FAT/SAT',
                          "Bachelor's degree in Electrical Engineering",
                          '10+ years of experience in project management and '
                          'electrical design'],
 'resume': 'ABDULLAH JAWAID \n'
           'Gulshan e Iqbal, block 2, Karachi, 75300 | +92 334 3664340 | '
           'abdullahjawaid@hotmail.com  \n'
           'Summary Statement \n'
           ' Initiative-taking Electrical Engineer with 10+ years of '
           'experience excelling in electrical design, project \n'
           'management, and Installation, Testing & Commissioning. Proven '
           'record in team engagement, \n'
           'equipment procurement, and project execution. Effective '
           'communicator with expertise in handling \n'
           'contractors, vendors, and clients. Proficient in progress '
           'reporting and adept at ensuring project success \n

## Train Baseline

In [3]:
from semantic_resume_matcher.classical.train_tfidf_logreg import train_tfidf_logreg

train_path = PROJECT_ROOT / "data" / "processed" / "resume_score_details.jsonl"
if not train_path.exists():
    train_path = PROJECT_ROOT / "data" / "raw" / "sample.jsonl"

output_dir = PROJECT_ROOT / "artifacts" / "tfidf_logreg"
config_path = PROJECT_ROOT / "configs" / "tfidf_logreg.yaml"

metrics = train_tfidf_logreg(train_path=train_path, output_dir=output_dir, config_path=config_path)
metrics

{'accuracy': 0.7016317016317016,
 'precision': 0.6995884773662552,
 'recall': 0.7555555555555555,
 'f1': 0.7264957264957265,
 'num_validation_examples': 429,
 'num_failures': 128}

## Inspect Validation Failures

In [4]:
import pandas as pd

failures_path = output_dir / "validation_failures.jsonl"
if failures_path.stat().st_size == 0:
    failures_df = pd.DataFrame(columns=["requirement", "label", "prediction", "probability", "resume_snippet"])
    failures_df
else:
    failures_df = pd.read_json(failures_path, lines=True)
    failures_df.assign(confidence=lambda df: (df["probability"] - 0.5).abs()).sort_values("confidence", ascending=False).head(20)

## Saved Artifacts

In [5]:
sorted(path.name for path in output_dir.iterdir())

['config.json',
 'metrics.json',
 'model.joblib',
 'validation_failures.jsonl',
 'validation_predictions.jsonl']